# Funcoes de ativacao da MLP

Este notebook mostra a implementacao das funcoes de ativacao **ReLU**, **Sigmoid** e **Softmax**, com explicacoes e exemplos praticos para MLP.


## Objetivo

- Implementar cada funcao com `numpy`.
- Mostrar como cada funcao transforma os valores de entrada.
- Explicar quando usar cada uma em redes MLP.
- Registrar fontes confiaveis (papers e documentacao oficial).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)


## 1) ReLU

Definicao:

\[
\mathrm{ReLU}(x)=\max(0, x)
\]

Intuicao:
- Zera valores negativos e mantem valores positivos.
- E uma escolha forte para camadas ocultas por simplicidade computacional e bom fluxo de gradiente na regiao positiva.

Contexto de literatura:
- Nair e Hinton (2010) mostraram ganhos com unidades retificadas em comparacao com unidades binarias em tarefas de visao [1].
- Glorot, Bordes e Bengio (2011) relataram desempenho igual ou superior dos retificadores em relacao a redes com ativacoes sigmoides/tanh em cenarios estudados [2].


In [ ]:
def relu(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    return np.maximum(0.0, x)

x_demo = np.array([-3.0, -1.0, 0.0, 0.5, 2.0])
print("Entrada:", x_demo)
print("ReLU:", relu(x_demo))


In [ ]:
x = np.linspace(-6, 6, 200)
y = relu(x)

plt.figure(figsize=(6, 4))
plt.plot(x, y, label="ReLU(x) = max(0, x)")
plt.axhline(0, color="gray", lw=1)
plt.axvline(0, color="gray", lw=1)
plt.title("Curva da ReLU")
plt.xlabel("x")
plt.ylabel("ReLU(x)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 2) Sigmoid

Definicao:

\[
\sigma(x)=\frac{1}{1+e^{-x}}
\]

Propriedades:
- Saida no intervalo \((0,1)\).
- Muito usada na **saida de classificacao binaria** por poder ser interpretada como probabilidade.

Observacao pratica:
- Para valores muito grandes em modulo, a sigmoid satura perto de 0 ou 1.


In [ ]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    # Clip para melhorar estabilidade numerica em exp()
    x = np.clip(x, -500.0, 500.0)
    return 1.0 / (1.0 + np.exp(-x))

x_demo = np.array([-10.0, -2.0, 0.0, 2.0, 10.0])
print("Entrada:", x_demo)
print("Sigmoid:", sigmoid(x_demo))


In [ ]:
x = np.linspace(-10, 10, 300)
y = sigmoid(x)

plt.figure(figsize=(6, 4))
plt.plot(x, y, label="sigmoid(x)")
plt.axhline(0.5, color="gray", lw=1, ls="--")
plt.title("Curva da Sigmoid")
plt.xlabel("x")
plt.ylabel("sigmoid(x)")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


## 3) Softmax

Definicao para um vetor \(z\):

\[
\mathrm{softmax}(z_i)=\frac{e^{z_i}}{\sum_j e^{z_j}}
\]

Propriedades:
- Converte logits em distribuicao de probabilidade por classe.
- Saidas entre 0 e 1 e soma igual a 1.
- Uso tipico na **saida de classificacao multiclasse**.

Implementacao estavel:
- Subtrair \(\max(z)\) antes do `exp` reduz risco de overflow numerico.


In [ ]:
def softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

logits = np.array([2.5, 0.5, -1.0])
probs = softmax(logits)

print("Logits:", logits)
print("Softmax:", probs)
print("Soma:", probs.sum())


In [ ]:
labels = ["Classe A", "Classe B", "Classe C"]
probs = softmax(np.array([2.5, 0.5, -1.0]))

plt.figure(figsize=(6, 4))
plt.bar(labels, probs)
plt.title("Probabilidades por classe (Softmax)")
plt.ylabel("Probabilidade")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.25)
plt.show()


## 4) Como escolher na MLP

Regra pratica usada no projeto:
- **Camadas ocultas**: ReLU (padrao)
- **Saida binaria**: Sigmoid
- **Saida multiclasse**: Softmax
- **Regressao**: saida linear (sem ativacao de probabilidade)


In [ ]:
def select_mlp_activations(task_type: str = "classification", n_classes: int = 2):
    if task_type not in {"classification", "regression"}:
        raise ValueError("task_type deve ser 'classification' ou 'regression'.")

    if task_type == "regression":
        return {
            "hidden_activation": "relu",
            "output_activation": "linear",
            "justification": "Regressao pede saida continua sem normalizacao em probabilidades."
        }

    if n_classes == 2:
        return {
            "hidden_activation": "relu",
            "output_activation": "sigmoid",
            "justification": "Classificacao binaria: sigmoid retorna probabilidade em [0, 1]."
        }

    return {
        "hidden_activation": "relu",
        "output_activation": "softmax",
        "justification": "Classificacao multiclasse: softmax produz distribuicao entre classes."
    }

print("Binaria:", select_mlp_activations("classification", n_classes=2))
print("Multiclasse:", select_mlp_activations("classification", n_classes=4))
print("Regressao:", select_mlp_activations("regression"))


## 5) Exemplo de integracao com sklearn (MLPClassifier)

No `scikit-learn`, a ativacao das camadas ocultas pode ser `relu` ou `logistic` (sigmoid).

Observacao:
- Para classificacao, o `MLPClassifier` otimiza log-loss e disponibiliza `predict_proba`.
- Na pratica, para binario usa-se leitura de probabilidade positiva; para multiclasse, distribuicao entre classes.


In [ ]:
try:
    from sklearn.neural_network import MLPClassifier

    model_relu = MLPClassifier(hidden_layer_sizes=(32,), activation="relu", random_state=42, max_iter=50)
    model_sigmoid = MLPClassifier(hidden_layer_sizes=(32,), activation="logistic", random_state=42, max_iter=50)

    print("Modelos criados com sucesso:")
    print("- hidden activation relu:", model_relu.activation)
    print("- hidden activation sigmoid/logistic:", model_sigmoid.activation)
except Exception as e:
    print("sklearn indisponivel no ambiente atual:", e)


## 6) Referencias (fontes confiaveis)

[1] Nair, V., Hinton, G. E. (2010). *Rectified Linear Units Improve Restricted Boltzmann Machines*. ICML.  
Link: https://www.cs.toronto.edu/~hinton/absps/reluICML.pdf

[2] Glorot, X., Bordes, A., Bengio, Y. (2011). *Deep Sparse Rectifier Neural Networks*. AISTATS (PMLR 15).  
Link: https://proceedings.mlr.press/v15/glorot11a.html

[3] Keras documentation - Activation functions (`relu`, `sigmoid`, `softmax`).  
Link: https://keras.io/api/layers/activations/

[4] Keras documentation - Softmax layer (formula estavel com subtracao do max).  
Link: https://keras.io/api/layers/activation_layers/softmax/

[5] scikit-learn documentation - `MLPClassifier` (ativacoes ocultas `relu` e `logistic`).  
Link: https://sklearn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html

[6] Goodfellow, I., Bengio, Y., Courville, A. (2016). *Deep Learning*. MIT Press.  
Site oficial: https://www.deeplearningbook.org/

Data de acesso: 23/02/2026.
